In [0]:
  %pip install -U bertopic nltk
  dbutils.library.restartPython()

In [0]:
%run ./helper

In [0]:
import pandas as pd
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.text("input_table_name", input_table_list[0])
dbutils.widgets.dropdown("write_mode", write_mode_list[0], write_mode_list)
dbutils.widgets.dropdown("execution_id", write_mode_list[0], write_mode_list)


dbutils.widgets.dropdown("cluster_model", cluster_model_list[0], cluster_model_list)
dbutils.widgets.dropdown("layer", "0", ["0","1"])
dbutils.widgets.dropdown("max_number_of_topics", "12", ["10","12","20"])


catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
input_table_name=dbutils.widgets.get("input_table_name")
cluster_model_name=dbutils.widgets.get("cluster_model")
layer=int(dbutils.widgets.get("layer"))
max_number_of_topics=int(dbutils.widgets.get("max_number_of_topics"))


write_mode=dbutils.widgets.get("write_mode")
from datetime import datetime
timestamp=pd.to_datetime(datetime.now())
execution_id_list=[]
if layer==0:
    execution_id = timestamp.strftime("%Y_%m_%d_%H_%M_%S")
else:
    execution_id_list=spark.table(f"{catalog}.{schema}.topic_info_local").select("execution_id").dropDuplicates().toPandas()["execution_id"].to_list()
    dbutils.widgets.dropdown("execution_id", execution_id_list[0], execution_id_list)
    execution_id = dbutils.widgets.get("execution_id")
    father_topic_list=spark.table(f"{catalog}.{schema}.topic_info_local").select("execution_id").dropDuplicates().toPandas()["execution_id"].to_list()
    dbutils.widgets.dropdown("father_topic", execution_id_list[0], execution_id_list)



In [0]:
timestamp

In [0]:
df = spark.read.table(f"{catalog}.{schema}.{input_table_name}").select("id", "text", "translated", "text_embedding")
table_name = f"{catalog}.{schema}.{input_table_name}"
print(f"table_name:{table_name}")


In [0]:
pddf=df.toPandas()

In [0]:
import numpy as np
embeddings = np.array(pddf["text_embedding"].tolist())
docs=pddf["text"].tolist()

In [0]:
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer


# 1. Load standard NLTK stop words
nltk.download('stopwords', quiet=True)
base_arabic_stopwords = nltk.corpus.stopwords.words('arabic')
base_english_stopwords = nltk.corpus.stopwords.words('english')

# 2. Consolidated Lebanese Dialect Stop Words
lebanese_dialect_stopwords = [
    "هيدا", "هيدي", "شو", "عم", "انتي", "انت", "يلي", "اللي", 
    "انو", "بس", "هيك", "مش", "رح", "شي", "مين", "كيف", "كان", 
    "صار", "ليش", "هون", "هلق", "يا", "ما", "في", "ولا", "وين", "اي",
    "بدك", "بدي", "الك", "بكون", "اديش", "وانا"
]

# 3. Consolidated Arabizi (Franco-Arabic) Stop Words
arabizi_stopwords = [
    "ya", "ma", "la", "fi", "bl", "3a", "mn", "rab", "allah", 
    "el", "al", "w", "b", "shu", "hek", "kif", "min", "y²", "x²",
    "kel", "3am", "ana", "3al", "bi", "bas", "bel", "sho"
]

# 4. Consolidated Tokenization Artifacts & Noise
artifact_stopwords = [
    "ال", "الل", "الم", "الن", "الح", "اج", "اس", "ؤك", "خر", 
    "عال", "را", "ام", "اب", "ان", "او", "الى", "الا",
    "الس", "هاب", "اض", "اء", "وال", "وآل", "آل"
]

# 5. Combine everything and remove any duplicates
all_custom_stopwords = list(set(
    base_arabic_stopwords + 
    base_english_stopwords + 
    lebanese_dialect_stopwords + 
    arabizi_stopwords + 
    artifact_stopwords
))
# Optional: Deduplicate the list
all_custom_stopwords = list(set(all_custom_stopwords))

# 6. Pass the combined list to your CountVectorizer
vectorizer_model = CountVectorizer(stop_words=all_custom_stopwords)




In [0]:
# 1. Initialize the CountVectorizer with Arabic stop words
vectorizer_model = CountVectorizer(stop_words=all_custom_stopwords)

# 2. (Optional but recommended) Enable BM25 weighting to reduce frequent word dominance
ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)


In [0]:
from hdbscan import HDBSCAN
from umap import UMAP
from bertopic import BERTopic
from sklearn.cluster import KMeans

# 1. Dimensionality Reduction (Crucial for HDBSCAN)
# HDBSCAN struggles with 768-dimension vectors, so we use UMAP to shrink them down to 5 dimensions first.
print("Configuring UMAP...")
umap_model = UMAP(n_neighbors=10, n_components=12, min_dist=0.0, metric='cosine', random_state=42)

# 2. Configure HDBSCAN
# We set min_cluster_size=100 to force the model to only create broad, massive topics.
print("Configuring HDBSCAN...")
cluster_model=None
if cluster_model_name=="KMEANS":
    cluster_model = KMeans(n_clusters=max_number_of_topics, random_state=42)
else:
    cluster_model = HDBSCAN(
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# 3. Initialize BERTopic with your custom models
print("Initializing BERTopic...")
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    language="arabic"
)

# 4. Fit the model using your PRE-CALCULATED MARBERTv2 embeddings
# This skips the heavy transformer step and runs in seconds!
print("Fitting the model...")
topics, probabilities = topic_model.fit_transform(docs, embeddings=embeddings)

# 5. View your new, generalized topics
print("Top Topics Discovered:")
print(topic_model.get_topic_info().head(12))

In [0]:
topic_model.reduce_topics(docs, nr_topics=max_number_of_topics)

In [0]:
topic_model.get_topic_info()

In [0]:
import pandas as pd
documents_df = pd.DataFrame({
    "Document": docs, 
    "ID": range(len(docs)),
    "Topic": topic_model.topics_
})

# 2. Re-calculate representative documents with a higher limit
extracted = topic_model._extract_representative_docs(
    c_tf_idf=topic_model.c_tf_idf_,
    documents=documents_df,
    topics=topic_model.topic_representations_,
    nr_samples=500,    
    nr_repr_docs=50,
    diversity=0.3    
)

# 3. Update the model's internal dictionary
# The first element in the returned tuple contains the {topic_id: [list_of_docs]} mapping
topic_model.representative_docs_ = extracted[0]

# 4. Verify the changes
topic_info = topic_model.get_topic_info()
print(topic_info[["Topic", "Representative_Docs"]].head())

In [0]:
topic_pd=topic_model.get_topic_info()

In [0]:
topic_pd

In [0]:
topic_pd=topic_pd.rename(
    columns={
        "Topic": "topic",
        "Count": "count",
        "Name": "name",
        "Representation": "representation",
        "Representative_Docs": "representative_docs"
    }
)
topic_pd["layer"]=layer
topic_pd["model"]=cluster_model_name
topic_pd["timestamp"]=timestamp
topic_pd["execution_id"]=execution_id


In [0]:
topic_info_df=spark.createDataFrame(topic_pd)


In [0]:
topic_info_df.display()

In [0]:
write_to_table(f"{catalog}.{schema}.topic_info_local", topic_info_df)


In [0]:
pddf_with_topic=pddf[["id"]]
pddf_with_topic["model"]=cluster_model_name
pddf_with_topic["timestamp"]=timestamp
pddf_with_topic["topic"]=topics
pddf_with_topic["layer"]=layer
pddf_with_topic["execution_id"]=execution_id
if layer=="0":
    pddf_with_topic["father_topic"]=topics
table_name = f"{catalog}.{schema}.topics"
df = spark.createDataFrame(pddf_with_topic)
write_to_table(table_name, df)

In [0]:
df.display()

In [0]:
table_info = spark.sql(f"DESCRIBE DETAIL {catalog}.{schema}.topic_info_local").collect()[0]
properties = table_info['properties']

if properties.get('delta.enableChangeDataFeed') == 'true':
    print("✅ CDF is enabled.")
else:
    print("❌ CDF is NOT enabled.")